# H-005 Baseline Experiment

WP-3c baseline notebook. This notebook reproduces the baseline metrics from repo-truth data and helpers. It does not append to `research/trial_log.jsonl` when rerun. The committed trial ID is `f4605dfa-fdad-4b76-8bbf-6e0a805d42a4`.


Evaluation note: no separate IS/OOS split convention exists in repo truth. The baseline uses the full registered evaluation range for both `in_sample_sharpe` and `oos_sharpe`, matching the current runbook baseline pattern.

Gate 2 semantics note: this notebook reports Gate 2 according to current `lib/evaluation.py` repo truth. The displayed legacy `GateReport` wording may read `DSR > 1.0 at 95% CI`, but the evaluator currently checks whether `dsr_val` exceeds 0.95 against its expected maximum-SR benchmark from the supplied trial count. Interpret the Gate 2 result according to the evaluator implementation, not as proof that the strategy met a true Sharpe > 1.0 standard.

Data note: WP-3c uses the catalogued 1h OHLCV parquet directly; no derived data is persisted. Full funding history is loaded, then reduced to the funding events that can apply while the generated signal is in position before calling `run_backtest()`. This preserves funding cashflows while keeping intraday notebook execution tractable because the current backtester scans all supplied funding events on each open-position bar.


In [ ]:
from pathlib import Path
import json
import math
import pandas as pd
from lib.backtest import run_backtest
from lib.evaluation import check_promotion_gates, cost_drag, hit_rate, max_drawdown, profit_factor, sharpe
from lib.evaluation import _sha256_of_dict

FUNDING_PATH = Path("data/processed/binance_um_perp_BTCUSDT_funding_8h_2019-09-10_2026-04-12_2026-04-12T1655Z.parquet")
COSTS = {"fee_bps_taker": 4.0, "fee_bps_maker": 2.0, "slippage_bps": 2.0, "initial_equity": 10000.0}

def load_ohlcv(path):
    df = pd.read_parquet(path).copy()
    df.index = pd.to_datetime(df["close_time_ms"].astype("int64"), unit="ms", utc=True)
    return df[["open", "high", "low", "close", "volume"]].astype(float).sort_index()

def load_funding():
    df = pd.read_parquet(FUNDING_PATH).copy()
    df.index = pd.to_datetime(df["fundingTime"].astype("int64"), unit="ms", utc=True)
    return df.rename(columns={"fundingRate": "funding_rate"})[["funding_rate"]].astype(float).sort_index()

def filter_applicable_funding(signal, ohlcv, funding):
    """Keep only funding events that can apply while the generated strategy is in-position.

    run_backtest() checks every supplied funding event on every open-position bar.
    Filtering irrelevant flat-period events preserves cashflows while keeping intraday
    notebooks runnable without changing lib/backtest.py.
    """
    common_idx = signal.index.intersection(ohlcv.index)
    sig = signal.reindex(common_idx).fillna(0).astype(int)
    bars = list(common_idx)
    position = 0
    event_ts = []
    funding_idx = funding.index
    for i, bar_ts in enumerate(bars):
        if i > 0 and position != 0:
            prev_bar_ts = bars[i - 1]
            matches = funding_idx[(funding_idx > prev_bar_ts) & (funding_idx <= bar_ts)]
            if len(matches):
                event_ts.extend(matches)
        if i > 0:
            desired = int(sig.iloc[i - 1])
            if desired != position:
                position = desired
    if not event_ts:
        return funding.iloc[0:0].copy()
    return funding.loc[pd.DatetimeIndex(sorted(set(event_ts)))]

def run_candidate(signal, ohlcv, funding, interval):
    applicable_funding = filter_applicable_funding(signal, ohlcv, funding)
    result = run_backtest(signal, ohlcv, applicable_funding, **COSTS, execution="next_bar_open", stream="klines", interval=interval)
    return result, len(applicable_funding)

def summarize(result, freq):
    returns = result.equity_curve.pct_change().dropna()
    mdd, peak, trough = max_drawdown(result.equity_curve)
    cd = cost_drag(result)
    return {
        "final_equity": float(result.equity_curve.iloc[-1]),
        "total_pnl": float(result.equity_curve.iloc[-1] - result.equity_curve.iloc[0]),
        "sharpe": float(sharpe(returns, freq)),
        "max_drawdown": float(mdd),
        "max_drawdown_peak": str(peak) if peak is not None else None,
        "max_drawdown_trough": str(trough) if trough is not None else None,
        "n_obs": int(len(returns)),
        "n_trades": int(len(result.trades_df)),
        "hit_rate": float(hit_rate(result.trades_df)),
        "profit_factor": float(profit_factor(result.trades_df)),
        "fees_paid": float(result.fees_paid),
        "slippage_paid": float(result.slippage_paid),
        "funding_paid": float(result.funding_paid),
        "gross_pnl": float(cd["gross_pnl"]),
        "cost_drag_fraction": float(cd["cost_drag_fraction"]) if math.isfinite(cd["cost_drag_fraction"]) else None,
        "skipped_bars": int(result.skipped_bars),
    }

def side_summary(result):
    if result.trades_df.empty:
        return []
    return [
        {"direction": int(d), "count": int(len(g)), "net_pnl_sum": float(g["net_pnl"].sum()), "net_pnl_mean": float(g["net_pnl"].mean())}
        for d, g in result.trades_df.groupby("direction")
    ]

OHLCV_PATH = Path("data/processed/binance_um_perp_BTCUSDT_1h_2020-01-01_2026-04-11_2026-04-12T1655Z.parquet")
DATASET_IDS = [
    "binance_um_perp_BTCUSDT_1h_2020-01-01_2026-04-11_2026-04-12T1655Z",
    "binance_um_perp_BTCUSDT_funding_8h_2019-09-10_2026-04-12_2026-04-12T1655Z",
]

def h005_signal(ohlcv, macd_fast=12, macd_slow=26, macd_signal=9, swing_lookback=10, risk_reward_ratio=3):
    close = ohlcv["close"]
    macd_line = close.ewm(span=int(round(macd_fast)), adjust=False).mean() - close.ewm(span=int(round(macd_slow)), adjust=False).mean()
    signal_line = macd_line.ewm(span=int(round(macd_signal)), adjust=False).mean()
    swing_high = ohlcv["high"].shift(1).rolling(int(round(swing_lookback)), min_periods=int(round(swing_lookback))).max()
    swing_low = ohlcv["low"].shift(1).rolling(int(round(swing_lookback)), min_periods=int(round(swing_lookback))).min()
    cross_above = (macd_line.shift(1) <= signal_line.shift(1)) & (macd_line > signal_line)
    cross_below = (macd_line.shift(1) >= signal_line.shift(1)) & (macd_line < signal_line)
    signal = pd.Series(0.0, index=ohlcv.index, name="signal")
    current = 0.0
    target = None
    for ts in ohlcv.index:
        if pd.isna(swing_high.loc[ts]) or pd.isna(swing_low.loc[ts]):
            current = 0.0
            target = None
            signal.loc[ts] = current
            continue
        c = close.loc[ts]
        if current == 1.0 and (bool(cross_below.loc[ts]) or c < swing_low.loc[ts] or (target is not None and c >= target)):
            current = 0.0
            target = None
        elif current == -1.0 and (bool(cross_above.loc[ts]) or c > swing_high.loc[ts] or (target is not None and c <= target)):
            current = 0.0
            target = None
        if current == 0.0:
            if bool(cross_above.loc[ts]) and c > swing_high.loc[ts]:
                risk = c - swing_low.loc[ts]
                current = 1.0
                target = c + float(risk_reward_ratio) * risk if risk > 0 else None
            elif bool(cross_below.loc[ts]) and c < swing_low.loc[ts]:
                risk = swing_high.loc[ts] - c
                current = -1.0
                target = c - float(risk_reward_ratio) * risk if risk > 0 else None
        signal.loc[ts] = current
    return signal

def sensitivity_h005(ohlcv, funding, base):
    variants = [
        ("macd_fast_up_25pct", {"macd_fast": 15, "macd_slow": 26, "macd_signal": 9, "swing_lookback": 10, "risk_reward_ratio": 3}),
        ("macd_fast_down_25pct", {"macd_fast": 9, "macd_slow": 26, "macd_signal": 9, "swing_lookback": 10, "risk_reward_ratio": 3}),
        ("macd_slow_up_25pct", {"macd_fast": 12, "macd_slow": 32, "macd_signal": 9, "swing_lookback": 10, "risk_reward_ratio": 3}),
        ("macd_slow_down_25pct", {"macd_fast": 12, "macd_slow": 20, "macd_signal": 9, "swing_lookback": 10, "risk_reward_ratio": 3}),
        ("macd_signal_up_25pct", {"macd_fast": 12, "macd_slow": 26, "macd_signal": 11, "swing_lookback": 10, "risk_reward_ratio": 3}),
        ("macd_signal_down_25pct", {"macd_fast": 12, "macd_slow": 26, "macd_signal": 7, "swing_lookback": 10, "risk_reward_ratio": 3}),
        ("swing_lookback_up_25pct", {"macd_fast": 12, "macd_slow": 26, "macd_signal": 9, "swing_lookback": 12, "risk_reward_ratio": 3}),
        ("swing_lookback_down_25pct", {"macd_fast": 12, "macd_slow": 26, "macd_signal": 9, "swing_lookback": 8, "risk_reward_ratio": 3}),
        ("risk_reward_ratio_up_25pct", {"macd_fast": 12, "macd_slow": 26, "macd_signal": 9, "swing_lookback": 10, "risk_reward_ratio": 3.75}),
        ("risk_reward_ratio_down_25pct", {"macd_fast": 12, "macd_slow": 26, "macd_signal": 9, "swing_lookback": 10, "risk_reward_ratio": 2.25}),
    ]
    rows = []
    for name, params in variants:
        signal = h005_signal(ohlcv, **params)
        result, _ = run_candidate(signal, ohlcv, funding, "1h")
        sr = summarize(result, "1h")["sharpe"]
        rows.append({"variant": name, "sharpe": sr, "ratio_to_base": sr / base if base else 0.0})
    return rows

HYPOTHESIS_ID = "H-005"
TRIAL_ID = "f4605dfa-fdad-4b76-8bbf-6e0a805d42a4"
TRIAL_COUNT_BEFORE_APPEND = 4
PARAMS = {"macd_fast": 12, "macd_slow": 26, "macd_signal": 9, "swing_lookback": 10, "risk_reward_ratio": 3}
ohlcv = load_ohlcv(OHLCV_PATH)
funding = load_funding()
signal = h005_signal(ohlcv, **PARAMS)
result, funding_events_applied = run_candidate(signal, ohlcv, funding, "1h")
metrics = summarize(result, "1h")
sensitivity = sensitivity_h005(ohlcv, funding, metrics["sharpe"])
params_meta = {"n_free_params": 5, "sensitivity_min_ratio": min(r["ratio_to_base"] for r in sensitivity), "is_sharpe": metrics["sharpe"]}
gate_report = check_promotion_gates(result=result, trial_count=TRIAL_COUNT_BEFORE_APPEND, params_meta=params_meta, freq="1h")
config = {"hypothesis_id": HYPOTHESIS_ID, "primary_timeframe": "1h", "execution": "next_bar_open", "costs": COSTS, "params": PARAMS}
print(json.dumps({
    "hypothesis_id": HYPOTHESIS_ID,
    "trial_id": TRIAL_ID,
    "trial_count_before_append": TRIAL_COUNT_BEFORE_APPEND,
    "dataset_ids": DATASET_IDS,
    "dataset_sha256": _sha256_of_dict({"ids": sorted(DATASET_IDS)}),
    "config_sha256": _sha256_of_dict(config),
    "params": PARAMS,
    "funding_events_applied": funding_events_applied,
    "signal_counts": {str(k): int(v) for k, v in signal.value_counts().sort_index().items()},
    "metrics": metrics,
    "sensitivity": sensitivity,
    "sensitivity_min_ratio": params_meta["sensitivity_min_ratio"],
    "side_summary": side_summary(result),
    "gate_report": gate_report.to_dict(),
}, indent=2, sort_keys=True))
